In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap

df_obras = pd.read_csv('../../db/clean/obras-nao-publicitarias.csv')
df_fsa = pd.read_csv('../../db/cluster/obras_investimento_fsa.csv', sep=";")
df_coproducoes = pd.read_csv('../../db/cluster/coproducoes_brasileiras.csv', sep=";")
df_fomento_ind = pd.read_csv('../../db/cluster/obras_fomento_indireto.csv', sep=";")
df_lanc_comerciais = pd.read_csv('../../db/cluster/lancamentos_comerciais.csv', sep=";") # Remover?
df_dist = pd.read_csv('../../db/clean/bilheteria-distribuidoras.csv', sep=",")
df_exib = pd.read_csv('../../db/clean/bilheteria-exibidoras.csv', sep=",")
df_fsa.columns = df_fsa.columns.str.lower()
df_coproducoes.columns = df_coproducoes.columns.str.lower()
df_fomento_ind.columns = df_fomento_ind.columns.str.lower()
df_lanc_comerciais.columns = df_lanc_comerciais.columns.str.lower()

In [57]:
# Remove os filmes estrangeiros de `df_lanc_comerciais`
df_lanc_comerciais = df_lanc_comerciais[~df_lanc_comerciais['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_lanc_comerciais.rename(columns={'cpb_roe': 'cpb'}, inplace=True)


/tmp/ipykernel_225000/1637947490.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lanc_comerciais.rename(columns={'cpb_roe': 'cpb'}, inplace=True)


In [58]:
# Trata o campo `renda_total`
df_lanc_comerciais = df_lanc_comerciais.copy()
df_lanc_comerciais['renda_total'] = df_lanc_comerciais['renda_total'] \
.str.replace('R$ ', '', regex=False) \
.str.replace('.', '', regex=False) \
.str.replace(',', '.', regex=False) \
.astype(float)

In [59]:
# Remove os filmes estrangeiros de `df_dist`
df_dist = df_dist[~df_dist['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_dist.rename(columns={'cpb_roe': 'cpb'}, inplace=True)

In [60]:
# Remove os filmes estrangeiros de `df_exib`
df_exib = df_exib[~df_exib['cpb_roe'].str.startswith('E', na=False)]

# Renomeia a coluna `cpb_roe` apenas para `cpb`, já que não há mais registros estrangeiros
df_exib.rename(columns={'cpb_roe': 'cpb'}, inplace=True)

# Renomeia coluna `sessao`
df_exib.rename(columns={'sessao': 'data_exibicao'}, inplace=True)

In [61]:
# Remove registros do tipo "não classificado" do dataframe de obras
df_obras = df_obras[df_obras['tipo_obra'] != 'NÃO CLASSIFICADA']

In [62]:
df_cluster = pd.DataFrame()
df_cluster = df_obras[['cpb']].copy()
df_cluster.set_index('cpb', inplace=True)
df_cluster['investimento_fsa'] = pd.NA
df_cluster['coproducao'] = pd.NA
df_cluster['fomento_indireto'] = pd.NA
df_cluster['publico_total'] = pd.NA
df_cluster['qtd_sessoes'] = pd.NA
df_cluster['dias_cartaz'] = pd.NA
df_cluster['renda_total'] = pd.NA
df_cluster['independente'] = pd.NA

In [63]:
# Adiciona valores na coluna `investimento_fsa`
fsa_cpbs = set(df_fsa['cpb'].dropna())
df_cluster['investimento_fsa'] = df_cluster.index.isin(fsa_cpbs)

In [64]:
# Adiciona valores na coluna `coproducao`
coproducao_cpbs = set(df_coproducoes['cpb'].dropna())
df_cluster['coproducao'] = df_cluster.index.isin(coproducao_cpbs)

In [65]:
# Adiciona valores na coluna `fomento_indireto`
fomento_ind_cpbs = set(df_fomento_ind['cpb'].dropna())
df_cluster['fomento_indireto'] = df_cluster.index.isin(fomento_ind_cpbs)

In [66]:
# Adiciona colunas para `cat_duracao`
map_cats = df_obras.set_index('cpb')['cat_duracao']

df_cluster['cat_duracao'] = df_cluster.index.map(map_cats)

# df_cluster = pd.get_dummies(df_cluster, columns=['cat_duracao'])
# hot_encode_nomes = {
#     'cat_duracao_CURTA METRAGEM': 'curta_metragem',
#     'cat_duracao_MÉDIA METRAGEM': 'media_metragem',
#     'cat_duracao_LONGA METRAGEM': 'longa_metragem'
# }
# df_cluster = df_cluster.rename(columns=hot_encode_nomes)

In [67]:
df_obras['tipo_obra'].value_counts()

tipo_obra
VÍDEOMUSICAL    13106
DOCUMENTÁRIO     8991
FICÇÃO           8272
ANIMAÇÃO         1654
VARIEDADES        541
Name: count, dtype: int64

In [68]:
# Adiciona colunas para `tipo_obra`
map_tipo = df_obras.set_index('cpb')['tipo_obra']

df_cluster['tipo_obra'] = df_cluster.index.map(map_tipo)

df_cluster = pd.get_dummies(df_cluster, columns=['tipo_obra'])
hot_encode_nomes = {
    'tipo_obra_VÍDEOMUSICAL': 'videomusical',
    'tipo_obra_DOCUMENTÁRIO': 'documentario',
    'tipo_obra_FICÇÃO': 'ficcao',
    'tipo_obra_ANIMAÇÃO': 'animacao',
    'tipo_obra_VARIEDADES': 'variedades',
}
df_cluster = df_cluster.rename(columns=hot_encode_nomes)

In [69]:
# Adiciona a coluna `publico_total`
map_publico_comercial = df_lanc_comerciais.groupby('cpb')['publico_total'].sum()
map_publico_dist = df_dist.groupby('cpb')['publico'].sum()
map_publico_exib = df_exib.groupby('cpb')['publico'].sum()

s_comercial = map_publico_comercial.reindex(df_cluster.index)
s_dist = map_publico_dist.reindex(df_cluster.index)
s_exib = map_publico_exib.reindex(df_cluster.index)

df_cluster['publico_total'] = s_comercial.combine_first(s_dist).combine_first(s_exib)

In [70]:
# Adiciona a coluna `qtd_sessoes`
map_qtd_dist = df_dist.groupby('cpb').size()
map_qtd_exib = df_exib.groupby('cpb').size()

s_qtd_dist = map_qtd_dist.reindex(df_cluster.index)
s_qtd_exib = map_qtd_exib.reindex(df_cluster.index)

df_cluster['qtd_sessoes'] = s_qtd_dist.combine_first(s_qtd_exib)

In [71]:
# Adiciona a coluna `dias_cartaz`
df_dist = df_dist.copy()
df_dist['data_exibicao'] = pd.to_datetime(df_dist['data_exibicao'])
df_exib['data_exibicao'] = pd.to_datetime(df_exib['data_exibicao'])

map_dias_dist = df_dist.groupby('cpb')['data_exibicao'].nunique()
map_dias_exib = df_exib.groupby('cpb')['data_exibicao'].nunique()

s_dias_dist = map_dias_dist.reindex(df_cluster.index)
s_dias_exib = map_dias_exib.reindex(df_cluster.index)

df_cluster['dias_cartaz'] = s_dias_dist.combine_first(s_dias_exib)

In [72]:
# Adiciona a coluna `renda_total`
map_rendas = df_lanc_comerciais.groupby('cpb')['renda_total'].sum()

s_map_rendas = map_rendas.reindex(df_cluster.index)
df_cluster['renda_total'] = s_map_rendas

In [73]:
# Adiciona a coluna `independente`
filtro_indep = df_obras['classificacao_obra'] == 'BRASILEIRA INDEPENDENTE CONSTITUINTE DE ESPAÇO QUALIFICADO'
set_independente = set(df_obras.loc[filtro_indep, 'cpb'].dropna())
df_cluster['independente'] = df_cluster.index.isin(set_independente)

In [74]:
# df_cluster = df_cluster.dropna()

In [75]:
df_cluster

,investimento_fsa,coproducao,fomento_indireto,publico_total,qtd_sessoes,dias_cartaz,renda_total,independente,cat_duracao,animacao,documentario,ficcao,variedades,videomusical
cpb,,,,,,,,,,,,,,
B0901120600000,False,False,False,986.0,NaN,NaN,8970.00,True,LONGA METRAGEM,False,False,True,False,False
B0901024500000,False,False,False,2313.0,NaN,NaN,23001.98,False,LONGA METRAGEM,False,True,False,False,False
B0901028800000,False,False,True,1718.0,NaN,NaN,14936.00,True,LONGA METRAGEM,False,True,False,False,False
B0901003800000,False,False,False,NaN,NaN,NaN,NaN,False,LONGA METRAGEM,False,False,False,False,True
B0901057400000,False,False,False,NaN,NaN,NaN,NaN,False,CURTA METRAGEM,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
B2500249300000,True,False,False,NaN,NaN,NaN,NaN,True,LONGA METRAGEM,False,True,False,False,False
B2500310100000,False,False,False,NaN,NaN,NaN,NaN,True,CURTA METRAGEM,False,True,False,False,False
B2500143000000,False,False,False,NaN,NaN,NaN,NaN,False,CURTA METRAGEM,False,True,False,False,False


In [76]:
# ...existing code...
# Conta quantos nulos existem em cada linha
nulos_por_linha = df_cluster.isnull().sum(axis=1)

# Filtra as linhas que têm exatamente 1 nulo
registros_um_nulo = df_cluster[nulos_por_linha == 1]

# Exibe o resultado
registros_um_nulo
# ...existing code...

,investimento_fsa,coproducao,fomento_indireto,publico_total,qtd_sessoes,dias_cartaz,renda_total,independente,cat_duracao,animacao,documentario,ficcao,variedades,videomusical
cpb,,,,,,,,,,,,,,
B1001299300000,False,False,True,0.0,14.0,1.0,NaN,False,LONGA METRAGEM,False,True,False,False,False
B1001271800000,False,False,False,195.0,75.0,36.0,NaN,True,MÉDIA METRAGEM,False,True,False,False,False
B1001280400000,False,False,False,0.0,1.0,1.0,NaN,True,MÉDIA METRAGEM,False,True,False,False,False
B1001165300000,False,False,False,37.0,6.0,6.0,NaN,True,MÉDIA METRAGEM,False,True,False,False,False
B1101383500000,False,False,False,0.0,1.0,1.0,NaN,True,LONGA METRAGEM,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
B2500009800000,False,False,False,33.0,1.0,1.0,NaN,True,LONGA METRAGEM,False,False,True,False,False
B2500021300000,False,False,False,7.0,7.0,7.0,NaN,True,CURTA METRAGEM,False,False,False,False,True
B2500180400000,False,False,False,1152.0,17.0,9.0,NaN,False,LONGA METRAGEM,False,False,False,False,True


In [ ]:
# Exportando DataFrame para .csv tratado
# df_cluster.to_csv("../../db/cluster/df_cluster.csv", index=False)